# Análise dos dados do Zenodo (UFRGS/IPH)

Fonte: *Banco de dados das cheias na Região Hidrográfica do Lago Guaíba*, v1.2 —
https://zenodo.org/records/13227745 (CC-BY-4.0). Arquivo baixado por `scripts/coleta_zenodo.py`.

Perguntas:
1. Quais bairros de Porto Alegre alagaram em 2024 e nas cheias anteriores?
2. A simulação do INCT concorda com as manchas de inundação do Zenodo?
3. A água medida em campo em maio de 2024 bate com a mancha simulada?

Toda alteração feita nos dados está comentada no código com **ALTERAÇÃO**.

## 1. Carregar os dados

In [ ]:
import json
import re
import unicodedata
from pathlib import Path

import geopandas as gpd
import pandas as pd
import plotly.express as px

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ORIGINAIS = RAIZ / "dados" / "originais"
GPKG = ORIGINAIS / "zenodo" / "cheias_rhguaiba_2024_db_v12.gpkg"

# Bairros de Porto Alegre (camada do GeoPackage)
bairros = gpd.read_file(GPKG, layer="poa_admin_bairros")

# Simulação do INCT em formato longo (uma linha por nível e bairro)
with open(ORIGINAIS / "INCT" / "data.json", encoding="utf-8") as f:
    raw = json.load(f)
inct = pd.DataFrame([
    {"nivel_m": float(nivel.removesuffix("m").replace("_", ".")), "bairro": bairro, **valores}
    for nivel, bs in raw.items()
    for bairro, valores in bs.items()
])
print("Bairros no GeoPackage:", len(bairros), "| CRS:", bairros.crs)
print("Bairros no INCT:", inct["bairro"].nunique())

## 2. Limpeza: padronizar os nomes dos bairros

Problema de **formatos inconsistentes**: o GeoPackage escreve `PASSO DAREIA`, o INCT `Passo D'Areia`.
Para juntar as duas bases criamos uma coluna `chave` com o nome sem acento, sem apóstrofo e em minúsculas.

In [ ]:
def padroniza(nome):
    """Remove acentos e apóstrofos, deixa em minúsculas e com espaços simples."""
    sem_acento = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    sem_apostrofo = sem_acento.replace("'", "")
    return " ".join(re.sub(r"[^a-z0-9]+", " ", sem_apostrofo.lower()).split())

# ALTERAÇÃO: dois bairros têm nome diferente nas duas fontes; trocamos o nome do GeoPackage
# pelo nome usado no INCT (conferido manualmente no mapa de bairros).
NOMES_CORRIGIDOS = {"cel aparicio borges": "coronel aparicio borges", "sao jose": "vila sao jose"}

bairros["chave"] = bairros["nmbop"].map(padroniza).replace(NOMES_CORRIGIDOS)
inct["chave"] = inct["bairro"].map(padroniza)

so_no_inct = sorted(set(inct["chave"]) - set(bairros["chave"]))
print(f"Bairros do INCT sem polígono no GeoPackage ({len(so_no_inct)}):", so_no_inct)

**Dado faltante:** o GeoPackage usa a divisão antiga (83 bairros) e o INCT a nova, de 2016 (94 bairros).
Os 11 bairros acima não têm polígono, então **ficam fora das comparações** (estratégia: remover o dado).

## 3. % da área de cada bairro coberta por cada mancha de inundação

Manchas usadas (todas simuladas pelos autores sobre o relevo atual):

| Cheia | Nível do Guaíba | Camada |
|---|---|---|
| 1941 | 4,75 m | `poa_inundacao_1941_475cm` |
| 1967 | 3,13 m | `poa_inundacao_1967_313cm` |
| 2015 | 2,94 m | `poa_inundacao_2015_294cm` |
| set/2023 | 3,18 m | `poa_inundacao_2023_set_318cm` |
| nov/2023 | 3,46 m | `poa_inundacao_2023_nov_346cm` |
| 2024 | 5,30 m | `poa_inundacao_2024_530cm` |

In [ ]:
CHEIAS = {
    "1941": "poa_inundacao_1941_475cm",
    "1967": "poa_inundacao_1967_313cm",
    "2015": "poa_inundacao_2015_294cm",
    "set/2023": "poa_inundacao_2023_set_318cm",
    "nov/2023": "poa_inundacao_2023_nov_346cm",
    "2024": "poa_inundacao_2024_530cm",
}

# ALTERAÇÃO: os polígonos dos bairros têm coordenada Z (altitude), que não usamos; removemos com force_2d().
bairros["geometry"] = bairros.force_2d()
bairros["area_km2"] = bairros.area / 1e6  # o CRS é EPSG:31982 (metros), então a área sai em m²

def pct_alagado(camada):
    """% da área de cada bairro que fica dentro da mancha."""
    mancha = gpd.read_file(GPKG, layer=camada).to_crs(bairros.crs)
    # ALTERAÇÃO: make_valid() corrige polígonos com geometria inválida (ex.: bordas que se cruzam),
    # senão a interseção dá erro. union_all() junta todos os pedaços da mancha em um só polígono.
    mancha = mancha.make_valid().union_all()
    area_alagada_km2 = bairros.intersection(mancha).area / 1e6
    return (area_alagada_km2 / bairros["area_km2"] * 100).round(2)

alag = bairros[["chave", "nmbop", "area_km2"]].copy()
for cheia, camada in CHEIAS.items():
    alag[cheia] = pct_alagado(camada)

alag.sort_values("2024", ascending=False).head(15)

## 4. Estatística descritiva

Média, mediana, desvio padrão, **obliquidade** e **curtose** da % alagada por bairro.
O pandas usa as fórmulas ajustadas (Fisher-Pearson) e a curtose já vem como **excesso de curtose** (normal = 0).

In [ ]:
cheias = list(CHEIAS)
resumo = alag[cheias].describe().T[["mean", "50%", "std", "min", "max"]]
resumo["obliquidade"] = alag[cheias].skew()
resumo["curtose (excesso)"] = alag[cheias].kurt()
resumo.round(2)

In [ ]:
px.histogram(alag, x="2024", nbins=20, title="Distribuição da % da área alagada por bairro em 2024",
             labels={"2024": "% da área do bairro alagada"})

### Comparando com a distribuição normal

Para ver a assimetria, desenhamos sobre o histograma uma **curva normal com a mesma média e o mesmo desvio
padrão** dos dados, e marcamos a média e a mediana. Numa distribuição normal (simétrica) média = mediana
e o histograma acompanharia a curva.

In [ ]:
import numpy as np
import plotly.graph_objects as go

valores = alag["2024"]
media, desvio, mediana = valores.mean(), valores.std(), valores.median()

# Curva normal teórica com a mesma média e desvio padrão (apenas referência; não altera os dados)
x = np.linspace(-60, 110, 300)
normal = np.exp(-((x - media) ** 2) / (2 * desvio ** 2)) / (desvio * np.sqrt(2 * np.pi))

fig = go.Figure()
fig.add_histogram(x=valores, histnorm="probability density", xbins=dict(start=0, end=100, size=5),
                  name="bairros (dados reais)", marker_color="steelblue", opacity=0.7)
fig.add_scatter(x=x, y=normal, mode="lines", name="normal de referência", line=dict(color="gray", dash="dash"))
fig.add_vline(x=media, line_color="red", annotation_text=f"média = {media:.1f}%")
fig.add_vline(x=mediana, line_color="green", annotation_text=f"mediana = {mediana:.1f}%",
              annotation_position="bottom right")
fig.update_layout(title=f"% alagada por bairro em 2024 — obliquidade = {valores.skew():.2f}, "
                        f"curtose (excesso) = {valores.kurt():.2f}",
                  xaxis_title="% da área do bairro alagada", yaxis_title="densidade")
fig

**Leitura:** os dados **não** seguem a normal: há um pico enorme em 0% (bairros altos, que não alagam)
e uma **cauda longa à direita** (bairros da orla, que alagam quase inteiros). A média fica à direita da mediana —
é isso que a obliquidade positiva indica. A curva normal ainda "prevê" valores negativos, o que é impossível
para uma porcentagem: outro sinal de que a normal não descreve bem esses dados.

**Leitura:** a maioria dos bairros tem 0% (ficam em áreas altas) e poucos têm valores altos — distribuição
com **assimetria positiva** (obliquidade > 0, cauda à direita): a média fica acima da mediana.

### Outliers

Box plot da % alagada em cada cheia. Pontos fora das "caixas" são bairros muito diferentes dos demais.
Aqui eles **não são erro**: são os bairros da orla e das ilhas (ex.: Arquipélago), então **mantemos**.

In [ ]:
longo = alag.melt(id_vars="nmbop", value_vars=cheias, var_name="cheia", value_name="pct_alagada")
px.box(longo, x="cheia", y="pct_alagada", points="outliers", hover_name="nmbop",
       title="% da área alagada por bairro em cada cheia", labels={"pct_alagada": "% da área alagada"})

## 5. Correlação: cheias históricas × 2024 × INCT

Juntamos a % de prédios afetados do INCT a **5,3 m** (nível mais próximo da mancha de 2024, 5,30 m)
e calculamos a **matriz de correlação de Pearson**.

In [ ]:
inct_53 = inct.loc[inct["nivel_m"] == 5.3, ["chave", "bairro", "percentage"]]
inct_53 = inct_53.rename(columns={"percentage": "INCT 5,3m"})

# merge "inner": ficam só os 83 bairros que existem nas duas bases (ver seção 2)
comp = alag.merge(inct_53, on="chave", how="inner")
print(len(comp), "bairros comparados")

matriz = comp[cheias + ["INCT 5,3m"]].corr(method="pearson").round(2)
px.imshow(matriz, text_auto=True, color_continuous_scale="RdBu", range_color=(-1, 1),
          title="Matriz de correlação de Pearson")

Pearson mede só relação **linear**, então também olhamos o gráfico de dispersão (visualizar, não só a métrica):

In [ ]:
fig = px.scatter(comp, x="2024", y="INCT 5,3m", hover_name="bairro",
                 labels={"2024": "% da área alagada (mancha 2024, 5,30 m)",
                         "INCT 5,3m": "% de prédios afetados (INCT 5,3 m)"},
                 title="Cada ponto é um bairro")
fig.add_shape(type="line", x0=0, y0=0, x1=100, y1=100, line=dict(dash="dash", color="gray"))
fig

**Leitura:** correlação alta: o INCT e a mancha concordam sobre **quais** bairros alagam.
Mas % de área ≠ % de prédios (um parque alagado não tem prédios), então os valores não precisam ser iguais.
Correlação ≠ causalidade: as duas medidas sobem juntas porque dependem da mesma coisa — a altitude do bairro.

## 6. Discretização: classes de alagamento

Discretização **manual** (supervisionada por conhecimento do problema), com pontos de corte fáceis de explicar:

| Classe | % |
|---|---|
| Sem alagamento | até 1% |
| Baixo | 1% a 25% |
| Médio | 25% a 50% |
| Alto | acima de 50% |

Usamos 1% (e não 0%) como corte porque frações minúsculas vêm de bordas de polígonos encostando no rio.

In [ ]:
CORTES = [-0.01, 1, 25, 50, 100]
CLASSES = ["Sem alagamento", "Baixo", "Médio", "Alto"]

comp["classe_mancha_2024"] = pd.cut(comp["2024"], bins=CORTES, labels=CLASSES)
comp["classe_inct_53"] = pd.cut(comp["INCT 5,3m"], bins=CORTES, labels=CLASSES)

pd.crosstab(comp["classe_mancha_2024"], comp["classe_inct_53"],
            rownames=["Mancha 2024"], colnames=["INCT 5,3 m"])

A diagonal da tabela são os bairros em que as duas fontes dão a mesma classe.

## 7. Binarização e similaridade (Jaccard)

**Binarização por limiar:** bairro "alagou" (1) se mais de 1% da área ficou sob água; senão 0.

Para comparar duas cheias usamos o **coeficiente de Jaccard**: aqui o 0 ("não alagou") não interessa
(atributo binário **assimétrico**), então contamos só os bairros que alagaram em pelo menos uma das duas.

In [ ]:
LIMIAR = 1  # %
binario = (comp[cheias + ["INCT 5,3m"]] > LIMIAR).astype(int)

def jaccard(a, b):
    m11 = ((a == 1) & (b == 1)).sum()
    m10 = ((a == 1) & (b == 0)).sum()
    m01 = ((a == 0) & (b == 1)).sum()
    return m11 / (m01 + m10 + m11)

colunas = cheias + ["INCT 5,3m"]
sim = pd.DataFrame([[jaccard(binario[a], binario[b]) for b in colunas] for a in colunas],
                   index=colunas, columns=colunas).round(2)
px.imshow(sim, text_auto=True, color_continuous_scale="Blues", range_color=(0, 1),
          title="Similaridade de Jaccard entre os bairros alagados")

In [ ]:
binario["nmbop"] = comp["nmbop"]
binario["n_cheias"] = binario[cheias].sum(axis=1)
print("Quantas das 6 cheias alagaram cada bairro:")
print(binario["n_cheias"].value_counts().sort_index())

print("\nBairros que alagaram em 2024 mas em nenhuma das cheias recentes (1967, 2015, 2023):")
recentes = ["1967", "2015", "set/2023", "nov/2023"]
print(binario.loc[(binario["2024"] == 1) & (binario[recentes].sum(axis=1) == 0), "nmbop"].tolist())

## 8. Mancha observada por satélite × INCT

Até aqui a mancha de 2024 era **simulada**. A camada `rhguaiba_planet-skysat_inundacao_obs_20240506` é a mancha
**observada** em 06/05/2024 (imagens Planet/Skysat + pontos de campo), perto do pico de 5,35 m.
Ela é grande (cobre toda a região do Guaíba), por isso recortamos só a área de Porto Alegre.

In [ ]:
obs = gpd.read_file(GPKG, layer="rhguaiba_planet-skysat_inundacao_obs_20240506")
# ALTERAÇÃO: a camada está em graus (EPSG:4326); convertemos para o CRS dos bairros (metros)
# e removemos a coordenada Z, que não usamos.
obs = obs.to_crs(bairros.crs)
obs["geometry"] = obs.force_2d()
# ALTERAÇÃO: recortamos a mancha para o retângulo que envolve Porto Alegre (descarta o resto da bacia)
# e corrigimos geometrias inválidas antes de juntar tudo em um polígono.
obs = gpd.clip(obs.make_valid(), bairros.total_bounds).union_all()

comp["obs 2024"] = (bairros.intersection(obs).area / 1e6 / bairros["area_km2"] * 100).round(2).values
# (bairros e comp estão na mesma ordem: comp veio de alag, que é cópia de bairros)
comp[["obs 2024", "2024", "INCT 5,3m"]].corr(method="pearson").round(2)

In [ ]:
fig = px.scatter(comp, x="obs 2024", y="INCT 5,3m", hover_name="bairro",
                 labels={"obs 2024": "% da área alagada (observado 06/05/2024)",
                         "INCT 5,3m": "% de prédios afetados (INCT 5,3 m)"},
                 title="Observado em 2024 × simulação do INCT (cada ponto é um bairro)")
fig.add_shape(type="line", x0=0, y0=0, x1=100, y1=100, line=dict(dash="dash", color="gray"))
fig

In [ ]:
# Mesma discretização da seção 6
comp["classe_obs_2024"] = pd.cut(comp["obs 2024"], bins=CORTES, labels=CLASSES)
pd.crosstab(comp["classe_obs_2024"], comp["classe_inct_53"],
            rownames=["Observado 2024"], colnames=["INCT 5,3 m"])

In [ ]:
comp["diferenca_pp"] = (comp["INCT 5,3m"] - comp["obs 2024"]).round(1)
print("Bairros com maior diferença entre INCT e observado (pontos percentuais):")
comp.reindex(comp["diferenca_pp"].abs().sort_values(ascending=False).index)[
    ["bairro", "obs 2024", "2024", "INCT 5,3m", "diferenca_pp"]].head(10)

**Leitura:** diferença negativa = água cobriu mais área do que a % de prédios que o INCT dá como afetados
(ex.: áreas alagadas com poucos prédios); positiva = o INCT afeta mais prédios do que a área observada sugere.

## 9. Pontos de nível medidos em campo

Camada `poa_pontos_nivel`: pontos onde a equipe do GPDEN/IPH/UFRGS marcou **a borda da água** em maio de 2024,
com o nível do Guaíba no dia (`Nivel_G_m`).

In [ ]:
pontos = gpd.read_file(GPKG, layer="poa_pontos_nivel")
print(pontos.dtypes)
print("\nValores faltantes:\n", pontos.isna().sum())
print("\nLinhas duplicadas:", pontos.duplicated(subset=["X", "Y", "Data"]).sum())

In [ ]:
# ALTERAÇÃO: Nivel_G_m vem como texto; convertemos para número.
pontos["Nivel_G_m"] = pontos["Nivel_G_m"].astype(float)
# ALTERAÇÃO: os pontos estão em graus (EPSG:4326); convertemos para o CRS dos bairros (metros)
# para poder medir distâncias. force_2d() remove a coordenada Z, que não usamos.
pontos = pontos.to_crs(bairros.crs)
pontos["geometry"] = pontos.force_2d()

pontos.groupby(["Data", "Nivel_G_m"]).size().to_frame("pontos")

### Distância entre a borda medida e a borda da mancha simulada

Para cada ponto calculamos a **distância euclidiana** (em metros) até a borda da mancha de nível mais próximo
ao do dia. Quanto menor, mais a simulação acerta até onde a água chegou.

In [ ]:
# Mancha com nível mais próximo do nível medido em cada dia
MANCHA_DO_NIVEL = {
    4.65: "poa_inundacao_1941_475cm",      # 4,75 m
    4.95: "poa_inundacao_simulada_500cm",  # 5,00 m
    5.06: "poa_inundacao_simulada_500cm",  # 5,00 m
    5.29: "poa_inundacao_2024_530cm",      # 5,30 m
    5.33: "poa_inundacao_2024_530cm",      # 5,30 m
}
pontos["mancha"] = pontos["Nivel_G_m"].map(MANCHA_DO_NIVEL)

for camada in pontos["mancha"].unique():
    mancha = gpd.read_file(GPKG, layer=camada).to_crs(bairros.crs).make_valid().union_all()
    linhas = pontos["mancha"] == camada
    pontos.loc[linhas, "dentro_da_mancha"] = pontos.loc[linhas].within(mancha)
    pontos.loc[linhas, "dist_borda_m"] = pontos.loc[linhas].distance(mancha.boundary)

pontos.groupby(["Data", "Nivel_G_m", "mancha"]).agg(
    pontos=("dist_borda_m", "size"),
    dentro_da_mancha=("dentro_da_mancha", "mean"),
    dist_mediana_m=("dist_borda_m", "median"),
).round(2)

In [ ]:
px.box(pontos, x="Data", y="dist_borda_m", points="all", hover_name="Local",
       labels={"dist_borda_m": "distância até a borda da mancha (m)"},
       title="Distância entre a borda da água medida em campo e a borda simulada")

In [ ]:
p = pontos.to_crs(4326)
px.scatter_map(p, lat=p.geometry.y, lon=p.geometry.x, color="Data", hover_name="Local",
               hover_data=["Nivel_G_m", "dist_borda_m"], zoom=10.5, height=600,
               map_style="carto-positron", title="Pontos de borda da água medidos em campo")

**Leitura:**
* 05 e 06/05 (5,29–5,33 m): quase todos os pontos ficam dentro da mancha de 5,30 m, perto da borda.
* 08/05 (5,06 m, Sarandi): quase todos ficam **fora** da mancha de 5,00 m — a água real foi além da simulada.
* 12/05 (4,65 m): comparado com a mancha de 4,75 m (nível maior), então é esperado ficarem dentro.

## Limitações
* As manchas históricas são **simuladas** sobre o relevo atual, não observadas.
* % de área alagada e % de prédios afetados (INCT) são medidas diferentes.
* 11 bairros novos do INCT não têm polígono no GeoPackage e ficaram fora.